# ATAT LightCurveTransformer — Attention-Weights Analysis

Loads the same model/data as `LC_graphcompare.ipynb`, captures attention
weights from every encoder layer via a non-invasive `_sa_block` monkey-patch,
and visualizes them at the sample and class level.

Run cells in order. Cell 10 restores the patched model.

In [1]:
# Cell 0 — Imports & config
import warnings
warnings.filterwarnings('ignore')

import types
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from tqdm import tqdm

from src.exploringtools.InitClassifier import InitClassifier, ztf_order_classes
from src.exploringtools.InitDataloader import InitDataLoader
from src.layers.classifiers.MultimodalClassifier import MultimodalClassifier
from src.layers.transformer.ATAT import LightCurveTransformer
from src.utils.data.AlerceDictionaries import ZTF_TAXONOMY

PATH_1 = '/home/magdalena/rpos/pipeline/pipeline/training/lc_classifier_ztf/ATAT_ALeRCE/results/PRETRAIN/LC/class_baseline_2e4_v2_0/'
FP_DATASET = '/home/magdalena/Desktop/sambashare/H5_files/BY_PARTITION/200_FF.h5'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 16

CLASS_NAMES = list(ZTF_TAXONOMY().keys())
print('device:', DEVICE, '| classes:', len(CLASS_NAMES))

device: cuda:0 | classes: 22


In [2]:
# Cell 1 — Load model & data (mirrors LC_graphcompare.ipynb cell 3)
test = InitClassifier(
    path_to_config_yaml=PATH_1,
    model=LightCurveTransformer,
    classifier=MultimodalClassifier,
    use_lc=True,
    arg_key='lc',
)

backbone_od = test.create_ordered_dict(
    remove_if_in_key_list=['projection', 'classifier'],
    rename_keys=('model.', ''),
    checkpoint_name='classifier_ckpt',
)
classifier_od = test.create_ordered_dict(
    remove_if_in_key_list=['projection', 'model'],
    rename_keys=('classifier.', ''),
    checkpoint_name='classifier_ckpt',
)
test.load_backbone_weights(backbone_od)
test.load_classifier_weights(classifier_od)
test.args.datamodule.val_use_sampler = False

dl = InitDataLoader(
    update_dataset_path=FP_DATASET,
    update_batch_size=BATCH_SIZE,
    datamodule_args=test.args.datamodule,
)
dl.init_test_dataset()

test.backbone.eval().to(DEVICE)
test.classifier.eval().to(DEVICE)

num_layers = len(test.backbone.transformer_lc.layers)
num_heads = test.backbone.transformer_lc.layers[0].self_attn.num_heads
print(f'encoder layers: {num_layers} | attention heads/layer: {num_heads}')

Using checkpoint 9550.ckpt
Using checkpoint 9550.ckpt
     - Loaded backbone weights
     - Loaded classifier weights
encoder layers: 3 | attention heads/layer: 4


In [3]:
# Cell 2 — Attention-capture patching
#
# Two problems to solve:
#   1. nn.TransformerEncoderLayer calls self_attn with need_weights=False,
#      so attention matrices are discarded. Fix: patch _sa_block to force
#      need_weights=True and stash the (B, H, S, S) tensor per layer.
#   2. nn.TransformerEncoder has a C++ fast path that skips the Python
#      layer.forward entirely (so our _sa_block patch would never run and
#      the stores would be empty). Fix: replace encoder.forward with a
#      plain Python loop that calls each layer.forward -> _sa_block.
# Originals are saved so Cell 10 can restore them.

def install_attention_capture(encoder):
    per_layer = [[] for _ in encoder.layers]
    originals_sa = [layer._sa_block for layer in encoder.layers]
    original_encoder_forward = encoder.forward

    for i, layer in enumerate(encoder.layers):
        store = per_layer[i]
        _sa = layer.self_attn
        _do = layer.dropout1

        def _sa_block(self, x, attn_mask, key_padding_mask, is_causal=False,
                      _store=store, _sa=_sa, _do=_do):
            out, w = _sa(
                x, x, x,
                attn_mask=attn_mask,
                key_padding_mask=key_padding_mask,
                need_weights=True,
                average_attn_weights=False,
                is_causal=is_causal,
            )
            _store.append(w.detach().cpu())
            return _do(out)

        layer._sa_block = types.MethodType(_sa_block, layer)

    def _encoder_forward(self, src, mask=None, src_key_padding_mask=None, is_causal=None):
        x = src
        for layer in self.layers:
            try:
                x = layer(
                    x,
                    src_mask=mask,
                    src_key_padding_mask=src_key_padding_mask,
                    is_causal=is_causal if is_causal is not None else False,
                )
            except TypeError:
                x = layer(x, src_mask=mask, src_key_padding_mask=src_key_padding_mask)
        if self.norm is not None:
            x = self.norm(x)
        return x

    encoder.forward = types.MethodType(_encoder_forward, encoder)

    def remove():
        for layer, orig in zip(encoder.layers, originals_sa):
            layer._sa_block = orig
        encoder.forward = original_encoder_forward

    def reset():
        for s in per_layer:
            s.clear()

    return per_layer, reset, remove

# Guard so re-running this cell doesn't double-wrap
try:
    remove  # noqa: F821
    remove()
except NameError:
    pass

per_layer, reset, remove = install_attention_capture(test.backbone.transformer_lc)
print('attention capture installed on', num_layers, 'layers')

attention capture installed on 3 layers


In [4]:
# Cell 3 — Run one batch and capture attention
batch = next(iter(dl.test_dataset))
batch = {k: v.to(DEVICE) for k, v in batch.items()}
true_labels = batch['labels'].detach().cpu().numpy()

with torch.no_grad():
    # embedding_light_curve returns (x_mod, m_mod, t_mod) with CLS already prefixed.
    # m_mod is True=valid, shape (B, S+1, 1). Does NOT invoke the encoder,
    # so the attention store stays untouched.
    _, m_mod, _ = test.backbone.embedding_light_curve(
        x=batch['data'],
        t=batch['time'],
        mask=batch['mask'],
        metadata=batch.get('metadata'),
        features=batch.get('features'),
    )
    mask = m_mod.squeeze(-1).detach().cpu().numpy().astype(bool)  # (B, S+1)

    reset()
    emb = test.backbone(**batch)
    cls_out = test.classifier(emb)
    if isinstance(cls_out, dict):
        logits = cls_out.get('LC', next(iter(cls_out.values())))
    else:
        logits = cls_out
    pred_labels = logits.argmax(-1).detach().cpu().numpy()

# Each per_layer[i] holds one tensor of shape (B, H, S+1, S+1)
attn = torch.stack([torch.cat(s, 0) for s in per_layer], dim=0).numpy()
print('attn shape (L, B, H, S+1, S+1):', attn.shape)
print('mask shape (B, S+1):', mask.shape)
assert attn.shape[:3] == (num_layers, BATCH_SIZE, num_heads)
assert mask.shape == (BATCH_SIZE, attn.shape[-1])
print('valid tokens per sample:', mask.sum(axis=1))

ValueError: torch.cat(): expected a non-empty list of Tensors

In [ ]:
# Cell 4 — Visualization helpers

def _masked_copy(a2d, mask_row, mask_col):
    """Return a2d with padded rows/cols set to NaN for heatmap display."""
    out = a2d.astype(float).copy()
    out[~mask_row, :] = np.nan
    out[:, ~mask_col] = np.nan
    return out


def cls_to_obs(attn_arr):
    """CLS-token attention over observations: (L, B, H, S)."""
    return attn_arr[..., 0, 1:]


def masked_entropy(p, valid):
    """Shannon entropy of distribution p, renormalized over valid positions only."""
    p = np.where(valid, p, 0.0)
    s = p.sum()
    if s <= 0:
        return np.nan
    p = p / s
    p = np.where(p > 0, p, 1.0)  # avoid log(0); zeros contribute 0
    return float(-(p * np.log(p)).sum())


_cmap = plt.get_cmap('viridis').copy()
_cmap.set_bad('lightgray')

In [ ]:
# Cell 5 — Per-sample per-layer heatmap grid
# Change sample_idx to inspect a different light curve in the batch.
sample_idx = 0
row_mask = mask[sample_idx]

true_cls = CLASS_NAMES[int(true_labels[sample_idx])]
pred_cls = CLASS_NAMES[int(pred_labels[sample_idx])]
valid_n = int(row_mask.sum())

fig, axes = plt.subplots(
    num_layers, num_heads,
    figsize=(3 * num_heads, 2.8 * num_layers),
    squeeze=False,
)
for li in range(num_layers):
    for hi in range(num_heads):
        ax = axes[li][hi]
        a = _masked_copy(attn[li, sample_idx, hi], row_mask, row_mask)
        im = ax.imshow(a, cmap=_cmap, aspect='auto')
        ax.set_title(f'L{li} H{hi}', fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(
    f'sample {sample_idx} — true: {true_cls} | pred: {pred_cls} | valid tokens: {valid_n}',
    y=1.01,
)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label='attention weight')
plt.show()

In [ ]:
# Cell 6 — CLS-rollout: how much attention the CLS token pays to each observation
cls_attn = cls_to_obs(attn)  # (L, B, H, S)
obs_mask = mask[:, 1:]        # drop the CLS slot itself

sample_rollout = cls_attn[:, sample_idx, :, :].mean(axis=1)  # head-averaged: (L, S)
valid_obs = obs_mask[sample_idx]

fig, ax = plt.subplots(figsize=(10, 4))
x_idx = np.arange(sample_rollout.shape[1])
for li in range(num_layers):
    series = np.where(valid_obs, sample_rollout[li], np.nan)
    ax.plot(x_idx, series, label=f'layer {li}')
ax.set_xlabel('observation index')
ax.set_ylabel('CLS attention (head-averaged)')
ax.set_title(f'CLS rollout — sample {sample_idx} | true: {true_cls} | pred: {pred_cls}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Sanity: CLS-row attention should sum to ~1 over (CLS + valid observations) per layer.
row_sums = attn[:, sample_idx, :, 0, :].sum(axis=-1)  # (L, H)
print('CLS-row sums (layer, head):')
print(np.round(row_sums, 4))

In [ ]:
# Cell 7 — Class-aggregated CLS attention (last layer, head-averaged)
# Variable sequence lengths → bin valid observations into N_BINS normalized buckets.
USE_PREDICTED = False  # flip to True to group by model prediction instead of ground truth
N_BINS = 50
layer_idx = num_layers - 1  # last encoder layer by default

group_labels = pred_labels if USE_PREDICTED else true_labels
cls_layer = cls_attn[layer_idx].mean(axis=1)  # (B, S) head-averaged

def bin_to_normalized(attn_row, valid_row, n_bins):
    valid_idx = np.where(valid_row)[0]
    if valid_idx.size == 0:
        return np.full(n_bins, np.nan)
    pos = (valid_idx - valid_idx.min()) / max(valid_idx.max() - valid_idx.min(), 1)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    which = np.clip(np.digitize(pos, bins) - 1, 0, n_bins - 1)
    out = np.full(n_bins, np.nan)
    for b in range(n_bins):
        sel = which == b
        if sel.any():
            out[b] = attn_row[valid_idx[sel]].mean()
    return out


class_matrix = {}
for b in range(BATCH_SIZE):
    cname = CLASS_NAMES[int(group_labels[b])]
    row = bin_to_normalized(cls_layer[b], obs_mask[b], N_BINS)
    class_matrix.setdefault(cname, []).append(row)

ordered = [c for c in ztf_order_classes if c in class_matrix]
heat = np.stack([np.nanmean(np.stack(class_matrix[c], 0), axis=0) for c in ordered], 0)

fig, ax = plt.subplots(figsize=(10, max(3, 0.35 * len(ordered))))
im = ax.imshow(heat, aspect='auto', cmap=_cmap)
ax.set_yticks(range(len(ordered)))
ax.set_yticklabels(ordered)
ax.set_xlabel('normalized observation position (0 = earliest valid, 1 = latest)')
ax.set_title(
    f'CLS→observation attention by class — layer {layer_idx}, '
    f'{"predicted" if USE_PREDICTED else "true"} labels'
)
fig.colorbar(im, ax=ax, label='mean attention')
plt.tight_layout(); plt.show()

print('classes present in this batch:', ordered)

In [ ]:
# Cell 8 — Per-head focus via attention entropy of the CLS row
# Low entropy → head picks out a few observations. High entropy → diffuse.
records = []
for li in range(num_layers):
    for hi in range(num_heads):
        for bi in range(BATCH_SIZE):
            valid = obs_mask[bi]
            if not valid.any():
                continue
            e = masked_entropy(cls_attn[li, bi, hi], valid)
            records.append({'layer': li, 'head': hi, 'sample': bi, 'entropy': e})

ent_df = pd.DataFrame(records)

fig, ax = plt.subplots(figsize=(max(6, 1.2 * num_heads * num_layers), 4))
positions, labels, data = [], [], []
for li in range(num_layers):
    for hi in range(num_heads):
        vals = ent_df.query('layer == @li and head == @hi')['entropy'].values
        data.append(vals)
        positions.append(li * (num_heads + 1) + hi)
        labels.append(f'L{li}H{hi}')
ax.boxplot(data, positions=positions, widths=0.7, showfliers=False)
ax.set_xticks(positions); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('CLS-row entropy (nats)')
ax.set_title('Attention focus per (layer, head) — lower = more peaky')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

print(ent_df.groupby(['layer', 'head'])['entropy'].describe().round(3))

In [ ]:
# Cell 9 — (Optional) Accumulate CLS-row attention across many batches
# Stores only the CLS row (not full S×S) to keep memory small.
# Then re-renders the class-aggregated heatmap and the entropy boxplot over more data.
RUN_MULTIBATCH = False
MAX_BATCHES = 20

if RUN_MULTIBATCH:
    all_cls = [[] for _ in range(num_layers)]  # each: list of (B, H, S)
    all_obs_mask = []
    all_true = []
    all_pred = []

    for bi, batch_i in enumerate(tqdm(dl.test_dataset)):
        if bi >= MAX_BATCHES:
            break
        batch_i = {k: v.to(DEVICE) for k, v in batch_i.items()}
        with torch.no_grad():
            _, m_mod_i, _ = test.backbone.embedding_light_curve(
                x=batch_i['data'], t=batch_i['time'], mask=batch_i['mask'],
                metadata=batch_i.get('metadata'), features=batch_i.get('features'),
            )
            obs_mask_i = m_mod_i.squeeze(-1).detach().cpu().numpy().astype(bool)[:, 1:]
            reset()
            emb_i = test.backbone(**batch_i)
            out_i = test.classifier(emb_i)
            logits_i = out_i['LC'] if isinstance(out_i, dict) else out_i
            pred_i = logits_i.argmax(-1).detach().cpu().numpy()

        for li in range(num_layers):
            w = per_layer[li][0]                     # (B, H, S+1, S+1)
            all_cls[li].append(w[..., 0, 1:].numpy())  # CLS row → (B, H, S)
        all_obs_mask.append(obs_mask_i)
        all_true.append(batch_i['labels'].detach().cpu().numpy())
        all_pred.append(pred_i)

    big_cls = np.stack([np.concatenate(layer_bufs, 0) for layer_bufs in all_cls], 0)
    big_mask = np.concatenate(all_obs_mask, 0)
    big_true = np.concatenate(all_true, 0)
    big_pred = np.concatenate(all_pred, 0)
    print('big_cls (L, N, H, S):', big_cls.shape)

    # Class heatmap over larger sample
    layer_idx = num_layers - 1
    cls_layer_big = big_cls[layer_idx].mean(axis=1)
    class_matrix = {}
    grp = big_pred if USE_PREDICTED else big_true
    for i in range(cls_layer_big.shape[0]):
        cname = CLASS_NAMES[int(grp[i])]
        class_matrix.setdefault(cname, []).append(
            bin_to_normalized(cls_layer_big[i], big_mask[i], N_BINS)
        )
    ordered = [c for c in ztf_order_classes if c in class_matrix]
    heat = np.stack([np.nanmean(np.stack(class_matrix[c], 0), axis=0) for c in ordered], 0)
    fig, ax = plt.subplots(figsize=(10, max(3, 0.35 * len(ordered))))
    im = ax.imshow(heat, aspect='auto', cmap=_cmap)
    ax.set_yticks(range(len(ordered))); ax.set_yticklabels(ordered)
    ax.set_xlabel('normalized observation position')
    ax.set_title(f'CLS→obs attention by class — layer {layer_idx} — {cls_layer_big.shape[0]} samples')
    fig.colorbar(im, ax=ax, label='mean attention')
    plt.tight_layout(); plt.show()
else:
    print('Set RUN_MULTIBATCH = True to aggregate across more batches.')

In [ ]:
# Cell 10 — Teardown: restore original _sa_block methods
remove()

# Verify: a fresh forward pass should NOT write to the attention store.
reset()
with torch.no_grad():
    _ = test.backbone(**batch)
sizes = [len(s) for s in per_layer]
print('store sizes after teardown (should all be 0):', sizes)
assert all(x == 0 for x in sizes), 'teardown failed'
print('attention capture removed; model restored to original state.')